In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
import torch
from sentence_transformers import SentenceTransformer, util


patent_docs = [
    {
        "patent_no": "PT2028/000001",
        "title": "Makine Öğrenmesi Tabanlı Kestirimci Bakım Sistemi",
        "abstract": (
            "Bu buluş, endüstriyel üretim hatlarında kullanılan motor, pompa ve konveyör "
            "sistemlerinden alınan sensör verilerini işleyerek arıza olasılığını tahmin eden "
            "bir makine öğrenmesi modeli ve karar destek yazılımı ile ilgilidir."
        )
    },
    {
        "patent_no": "PT2028/000002",
        "title": "Lityum İyon Batarya Hücreleri İçin Termal Yönetim Modülü",
        "abstract": (
            "Buluş, elektrikli araç batarya paketlerinde hücre sıcaklıklarını dengeleyen, "
            "ısı değiştirici kanallar ve batarya yönetim sistemi ile birlikte çalışan "
            "kompakt bir termal yönetim modülünü kapsamaktadır."
        )
    },
    {
        "patent_no": "PT2028/000003",
        "title": "IoT Tabanlı Akıllı Sulama ve Toprak Nem İzleme Sistemi",
        "abstract": (
            "Bu sistem, tarımsal alanlarda toprak nem sensörleri, kablosuz haberleşme modülü "
            "ve bulut tabanlı veri analitiği kullanarak sulama zamanını otomatik olarak "
            "belirleyen bir hassas tarım çözümüdür."
        )
    },
    {
        "patent_no": "PT2028/000004",
        "title": "Görüntü İşleme Destekli Kalite Kontrol Cihazı",
        "abstract": (
            "Buluş, üretim hattındaki ürünleri kamera sensörü ile görüntüleyen, görüntü işleme "
            "algoritmalarıyla yüzey kusurlarını tespit eden ve kalite kontrol raporu oluşturan "
            "otomatik bir test cihazı ile ilgilidir."
        )
    },
    {
        "patent_no": "PT2028/000005",
        "title": "Kriptografik Kimlik Doğrulama İçeren Dijital Cüzdan Yöntemi",
        "abstract": (
            "Bu buluş, mobil ödeme sistemlerinde kullanıcı kimliğini doğrulamak için şifreleme, "
            "kriptografik anahtar yönetimi ve çok faktörlü kimlik doğrulama adımlarını içeren "
            "güvenli bir dijital cüzdan yöntemini açıklamaktadır."
        )
    },
    {
        "patent_no": "PT2028/000006",
        "title": "Robot Kol İçin Kuvvet Geri Beslemeli Manipülatör Kontrolü",
        "abstract": (
            "Buluş, endüstriyel robot kollarında hassas montaj işlemleri için kuvvet sensörü, "
            "servo motor ve gerçek zamanlı kontrol birimi kullanan geri beslemeli bir "
            "manipülatör kontrol sistemine ilişkindir."
        )
    },
    {
        "patent_no": "PT2028/000007",
        "title": "Güneş Paneli Verimliliğini Artıran İnce Film Kaplama",
        "abstract": (
            "Bu buluş, fotovoltaik güneş panellerinde ışık yansımasını azaltan ve enerji üretim "
            "verimliliğini artıran nano yapılı ince film kaplama malzemesi ve kaplama prosesi "
            "ile ilgilidir."
        )
    },
    {
        "patent_no": "PT2028/000008",
        "title": "Uzaktan Hasta İzleme İçin Giyilebilir Biyosensör Platformu",
        "abstract": (
            "Buluş, hastanın kalp ritmi, vücut sıcaklığı ve hareket verilerini ölçen giyilebilir "
            "biyosensörler ile bu verileri dijital sağlık platformuna ileten uzaktan hasta "
            "takip sistemini kapsamaktadır."
        )
    },
    {
        "patent_no": "PT2028/000009",
        "title": "5G Ağlarında Dinamik Bant Genişliği Tahsisi Yöntemi",
        "abstract": (
            "Bu yöntem, 5G baz istasyonlarında ağ trafiğini analiz ederek gecikme, servis kalitesi "
            "ve kullanıcı yoğunluğuna göre bant genişliğini dinamik biçimde tahsis eden bir "
            "ağ optimizasyon algoritmasını açıklamaktadır."
        )
    },
    {
        "patent_no": "PT2028/000010",
        "title": "Geri Dönüştürülmüş Polimer Kompozit Yapı Malzemesi",
        "abstract": (
            "Buluş, geri dönüştürülmüş plastik ve cam elyafı içeren polimer kompozit bir yapı "
            "malzemesinin üretim yöntemi ile bu malzemenin inşaat sektöründe yalıtım ve panel "
            "uygulamalarında kullanımına ilişkindir."
        )
    }
]

df = pd.DataFrame(patent_docs)


OUTPUT_FILE = "output.xlsx"
USE_GPU = True

MODEL_NAME = "intfloat/multilingual-e5-large-instruct"
BATCH_SIZE = 32
MAX_SEQ_LENGTH = 512


tech_taxonomy = [
    {
        "ana_kategori": "Yazılım, Veri ve Yapay Zeka",
        "alt_kategoriler": [
            {"isim": "Yazılım altyapısı ve platformlar", "keywords": "işletim sistemi, os, bulut bilişim, cloud, uygulama programlama arayüzü, api, sunucu, backend, veritabanı yönetim sistemi, dbms, sanallaştırma, container, konteyner, kubernetes, mikroservis"},
            {"isim": "Veri işleme, analitik ve optimizasyon", "keywords": "büyük veri, big data, veri madenciliği, sql, veri ambarı, etl, optimizasyon algoritması, istatistiksel analiz, kümeleme, sınıflandırma, karar destek, analitik"},
            {"isim": "Yapay zeka ve makine öğrenmesi", "keywords": "yapay zeka, ai, makine öğrenmesi, machine learning, derin öğrenme, sinir ağı, neural network, doğal dil işleme, nlp, bilgisayarlı görü, computer vision, destek vektör makineleri, svm, model eğitimi, tahmin modeli"},
            {"isim": "İnsan-makine etkileşimi ve arayüz teknolojileri", "keywords": "kullanıcı arayüzü, ui, kullanıcı deneyimi, ux, dokunmatik ekran, insan makine etkileşimi, hmi, sanal gerçeklik, vr, artırılmış gerçeklik, ar, sesli komut, jest kontrolü"},
            {"isim": "Siber güvenlik, kriptografi ve güvenli bilişim", "keywords": "siber güvenlik, kriptografi, şifreleme, encryption, kimlik doğrulama, authentication, yetkilendirme, authorization, veri gizliliği, ağ güvenliği, sızma tespiti, intrusion detection, zararlı yazılım, malware, rsa, blokzincir, blockchain"}
        ]
    },
    {
        "ana_kategori": "Elektrik, Elektronik ve Gömülü Sistemler",
        "alt_kategoriler": [
            {"isim": "Elektronik devreler ve donanım", "keywords": "baskılı devre kartı, pcb, direnç, resistor, kondansatör, capacitor, lehimleme, anakart, elektronik devre, donanım, analog devre, dijital devre"},
            {"isim": "Gömülü sistemler ve kontrol birimleri", "keywords": "gömülü sistem, embedded system, mikrodenetleyici, mcu, fpga, programlanabilir lojik, gerçek zamanlı işletim sistemi, rtos, gömülü yazılım, plc, kontrol birimi"},
            {"isim": "Sensörler ve algılama donanımları", "keywords": "sensör, algılayıcı, transdüser, dönüştürücü, ivmeölçer, accelerometer, jiroskop, gyroscope, lidar, radar, sıcaklık sensörü, optik sensör"},
            {"isim": "Güç elektroniği", "keywords": "güç elektroniği, invertör, inverter, konvertör, converter, batarya yönetim sistemi, bms, güç kaynağı, smps, doğrultucu, rectifier, transformatör, trafo"},
            {"isim": "Ses/görüntü edinimi ve elektronik sinyal işleme", "keywords": "dijital sinyal işleme, dsp, ses işleme, görüntü işleme, filtreleme, gürültü engelleme, noise cancellation, adc, analog sayısal dönüştürücü, dac, sayısal analog dönüştürücü"}
        ]
    },
    {
        "ana_kategori": "Yarı İletken ve Mikroelektronik",
        "alt_kategoriler": [
            {"isim": "Yarı iletken malzeme ve aygıt yapıları", "keywords": "yarı iletken, semiconductor, silisyum, silicon, galyum nitrür, gallium nitride, mosfet, transistör, diyot, wafer, gofret, epitaksi"},
            {"isim": "Entegre devre ve çip tasarımı", "keywords": "entegre devre, integrated circuit, çip, chip, yonga, asic, vlsi, mantık kapısı, logic gate, bellek çipi, system on chip, soc"},
            {"isim": "Mikroelektronik üretim, paketleme ve test", "keywords": "fotolitografi, lithography, aşındırma, etching, çip paketleme, packaging, tel bağlama, wire bonding, çip testi, wafer testi, dökümhane, foundry"}
        ]
    },
    {
        "ana_kategori": "Ağ, İletim ve Bağlantı Teknolojileri",
        "alt_kategoriler": [
            {"isim": "Kablosuz haberleşme sistemleri", "keywords": "kablosuz haberleşme, wireless communication, 5g, 6g, baz istasyonu, antenna, anten, wi-fi, wifi, bluetooth, rfid, hücresel ağ, cellular network"},
            {"isim": "Sabit ağ, fiber erişim ve haberleşme altyapıları", "keywords": "fiber optik, broadband, genişbant, router, yönlendirici, switch, ağ anahtarı, ethernet, kablolu iletişim, omurga ağı, erişim ağı"},
            {"isim": "Ağ yönetimi ve optimizasyon", "keywords": "ağ yönetimi, network management, bant genişliği tahsisi, routing protocol, yönlendirme protokolü, paket anahtarlama, packet switching, ağ gecikmesi, latency, qos, trafik mühendisliği"},
            {"isim": "IoT ve bağlantılı cihaz mimarileri", "keywords": "nesnelerin interneti, iot, bağlantılı cihaz, m2m, sensör ağı, uç bilişim, edge computing, lora, bağlantı mimarisi"},
            {"isim": "Uydu ve karasal olmayan ağ teknolojileri", "keywords": "uydu haberleşmesi, satellite communication, yer istasyonu, telemetri, uzay iletişimi, gps, gnss, alçak yörünge, leo"}
        ]
    },
    {
        "ana_kategori": "Optik, Fotonik ve Görüntüleme Teknolojileri",
        "alt_kategoriler": [
            {"isim": "Optik bileşenler ve optik sistemler", "keywords": "mercek, lens, prizma, ayna, mirror, dalga kılavuzu, waveguide, optik fiber, kırınım ağı, diffraction grating, yansıma, kırılma"},
            {"isim": "Lazer ve fotonik teknolojileri", "keywords": "lazer, laser, lazer diyot, fotonik, photonics, foton, photon, optoelektronik, optoelectronics, led, lazer kesim, lidar"},
            {"isim": "Optik algılama ve görüntüleme sistemleri", "keywords": "kamera sensörü, image sensor, ccd, cmos, mikroskop, microscope, teleskop, telescope, kızılötesi kamera, infrared camera, spektrometre, tomografi"}
        ]
    },
    {
        "ana_kategori": "Mekanik, Mekatronik ve Robotik Sistemler",
        "alt_kategoriler": [
            {"isim": "Makine elemanları, mekanik düzenekler ve yapısal sistemler", "keywords": "dişli, gear, rulman, bearing, şaft, shaft, yay, spring, mil, gövde, şasi, mekanik bağlantı, menteşe"},
            {"isim": "Hareket ve aktüasyon mekanizmaları", "keywords": "motor, aktüatör, actuator, piston, hidrolik silindir, pneumatic, pnömatik, servo motor, lineer hareket, redüktör"},
            {"isim": "Akışkan, pompa, vana ve basınçlı sistemler", "keywords": "pompa, pump, valf, valve, vana, kompresör, compressor, türbin, turbine, akışkanlar dinamiği, basınç regülatörü, nozul"},
            {"isim": "Termal sistemler ve ısıl proses teknolojileri", "keywords": "ısı değiştirici, heat exchanger, soğutucu, radyatör, endüstriyel fırın, termal yönetim, ısı yalıtımı, iklimlendirme, hvac"},
            {"isim": "İmalat yöntemleri, şekillendirme ve montaj teknolojileri", "keywords": "kaynak, welding, döküm, casting, talaşlı imalat, cnc, 3b yazıcı, 3d printer, eklemeli imalat, additive manufacturing, ekstrüzyon, kalıplama"},
            {"isim": "Robotik ve otonom fiziksel sistemler", "keywords": "robot kol, manipülatör, manipulator, otonom robot, endüstriyel robot, agv, dron, drone, kinematik, mobil robot"},
            {"isim": "Ölçüm, test ve kalibrasyon sistemleri", "keywords": "ölçüm, measurement, test cihazı, kalibrasyon, calibration, kalite kontrol, metroloji, metrology, doğrulama, muayene"}
        ]
    },
    {
        "ana_kategori": "Kimya, Malzeme ve Yüzey Teknolojileri",
        "alt_kategoriler": [
            {"isim": "Temel kimya ve kimyasal prosesler", "keywords": "katalizör, catalyst, sentez, synthesis, çözücü, solvent, reaktör, reactor, damıtma, distillation, polimerizasyon, kimyasal reaksiyon, asit, baz"},
            {"isim": "Polimer, plastik, kauçuk ve kompozit teknolojileri", "keywords": "polimer, polymer, plastik, elastomer, poliüretan, polyurethane, karbon fiber, carbon fiber, cam elyafı, termoplastik, kompozit malzeme"},
            {"isim": "Metal, seramik ve cam malzemeler", "keywords": "çelik, steel, alüminyum, aluminum, alaşım, alloy, metalürji, metallurgy, sinterleme, seramik, ceramic, refrakter, temperli cam"},
            {"isim": "Yüzey, kaplama ve ince film teknolojileri", "keywords": "kaplama, coating, galvaniz, pvd, cvd, anotlama, anodizing, korozyon önleyici kaplama, ince film, thin film, boya, yüzey pürüzlülüğü"},
            {"isim": "Nano, ileri ve fonksiyonel malzemeler", "keywords": "grafen, graphene, karbon nanotüp, carbon nanotube, metamalzeme, metamaterial, piezoelektrik, süperiletken, superconductor, şekil bellekli alaşım, nanoteknoloji"}
        ]
    },
    {
        "ana_kategori": "Biyoteknoloji ve Yaşam Bilimleri Teknolojileri",
        "alt_kategoriler": [
            {"isim": "Moleküler biyoloji ve genetik teknolojiler", "keywords": "dna, rna, crispr, gen düzenleme, gene editing, pcr, sekanslama, sequencing, biyobelirteç, biomarker, mutasyon"},
            {"isim": "Hücre, doku ve rejeneratif teknolojiler", "keywords": "kök hücre, stem cell, hücre kültürü, cell culture, doku mühendisliği, tissue engineering, biyobaskı, bioprinting, in vitro, organoid, rejeneratif tıp"},
            {"isim": "Farmasötik formülasyon ve ilaç geliştirme", "keywords": "farmasötik formülasyon, ilaç geliştirme, drug development, aktif farmasötik bileşen, api, etkin madde, ilaç taşıma sistemi, kontrollü salım, tablet, kapsül, aşı, farmakokinetik"},
            {"isim": "Biyoproses ve biyoprodüksiyon teknolojileri", "keywords": "fermantasyon, fermentation, biyoreaktör, bioreactor, enzim, enzyme, mikrobiyal üretim, saflaştırma, purification, kromatografi, chromatography"},
            {"isim": "Biyosensör ve biyolojik analiz teknolojileri", "keywords": "biyosensör, biosensor, antikor, antibody, elisa, mikroakışkan, microfluidic, çip üstü laboratuvar, lab on a chip, teşhis kiti, analizör"}
        ]
    },
    {
        "ana_kategori": "Enerji ve Çevre Teknolojileri",
        "alt_kategoriler": [
            {"isim": "Enerji üretim teknolojileri", "keywords": "güneş paneli, solar panel, rüzgar türbini, wind turbine, jeneratör, generator, yakıt hücresi, fuel cell, nükleer reaktör, fotovoltaik, hydroelectric, hidroelektrik"},
            {"isim": "Enerji depolama ve batarya teknolojileri", "keywords": "enerji depolama, battery, batarya, lityum iyon, lithium ion, batarya hücresi, akü, süperkapasitör, supercapacitor, anot, katot, enerji yoğunluğu"},
            {"isim": "Enerji dönüşüm, iletim ve dağıtım teknolojileri", "keywords": "elektrik şebekesi, power grid, trafo merkezi, substation, yüksek gerilim, akıllı şebeke, smart grid, iletim hattı, dağıtım hattı, enerji yönetimi"},
            {"isim": "Su arıtma, atık işleme ve geri kazanım teknolojileri", "keywords": "su arıtma, water treatment, filtrasyon, filtration, ters osmoz, reverse osmosis, atıksu, wastewater, geri dönüşüm, recycling, biyolojik arıtma, desalinasyon"},
            {"isim": "Emisyon, karbon ve çevresel izleme teknolojileri", "keywords": "karbon yakalama, carbon capture, egzoz gazı temizleme, hava kalitesi sensörü, kirlilik kontrolü, pollution control, sera gazı, emisyon izleme, çevresel izleme"}
        ]
    },
    {
        "ana_kategori": "Kuantum Teknolojileri",
        "alt_kategoriler": [
            {"isim": "Kuantum hesaplama ve bilgi işleme", "keywords": "kübit, qubit, kuantum bilgisayar, quantum computer, dolanıklık, entanglement, süperpozisyon, superposition, kuantum algoritması, kuantum kapısı, quantum gate"},
            {"isim": "Kuantum haberleşme ve kriptografi", "keywords": "kuantum anahtar dağıtımı, qkd, kuantum kriptografi, quantum cryptography, güvenli iletişim, kuantum ağı, quantum network, kriptografik protokol"},
            {"isim": "Kuantum algılama, ölçüm ve konumlama", "keywords": "kuantum sensörü, quantum sensor, atomik saat, atomic clock, manyetometre, magnetometer, yüksek hassasiyetli ölçüm, kuantum radarı, kuantum konumlama"}
        ]
    }
]


sector_taxonomy = [
    {
        "ana_kategori": "Sağlık, Tıp ve Yaşam Bilimleri",
        "alt_kategoriler": [
            {"isim": "Tanı, görüntüleme ve hasta izleme", "keywords": "manyetik rezonans, mr, mri, röntgen, x-ray, ultrason, ultrasound, ekg, ecg, hasta monitörizasyonu, in vitro tanı, diagnostik, teşhis cihazı"},
            {"isim": "Tedavi, cerrahi ve müdahale teknolojileri", "keywords": "cerrahi, surgery, neşter, kateter, catheter, stent, lazer ameliyatı, radyoterapi, radiotherapy, diyaliz, cerrahi robot"},
            {"isim": "İlaç, biyofarmasötik ve tedavi edici biyoteknoloji", "keywords": "ilaç, drug, biyofarmasötik, biopharmaceutical, kanser ilacı, aşı, vaccine, antibiyotik, monoklonal antikor, gen terapisi, eczacılık ürünü"},
            {"isim": "İmplant, protez ve rejeneratif tıp uygulamaları", "keywords": "implant, protez, prosthesis, kalp pili, pacemaker, ortopedik implant, yapay eklem, diş implantı, biyouyumlu malzeme, protez kol"},
            {"isim": "Rehabilitasyon, fizik tedavi ve yardımcı yaşam teknolojileri", "keywords": "rehabilitasyon, fizik tedavi, physiotherapy, tekerlekli sandalye, işitme cihazı, hearing aid, dış iskelet, exoskeleton, yürüteç"},
            {"isim": "Ağız, diş ve oral sağlık uygulamaları", "keywords": "ağız sağlığı, oral health, diş fırçası, ortodonti, orthodontics, diş teli, kanal tedavisi, diş dolgusu, diş beyazlatma"},
            {"isim": "Dijital sağlık ve uzaktan hasta takibi", "keywords": "dijital sağlık, digital health, teletıp, telemedicine, hasta takip yazılımı, dijital reçete, mobil sağlık uygulaması, giyilebilir ekg, uzaktan hasta izleme"}
        ]
    },
    {
        "ana_kategori": "Otomotiv, Mobilite ve Ulaşım",
        "alt_kategoriler": [
            {"isim": "Kara araçları ve otomotiv sistemleri", "keywords": "otomobil, automobile, kamyon, truck, fren sistemi, brake system, süspansiyon, direksiyon, steering, şanzıman, transmission, içten yanmalı motor"},
            {"isim": "Elektrikli ve hibrit araç sistemleri", "keywords": "elektrikli araç, ev, electric vehicle, hibrit araç, hev, hybrid vehicle, elektrikli motor, araç şarj istasyonu, batarya paketi, menzil uzatıcı"},
            {"isim": "Otonom sürüş ve sürüş destek uygulamaları", "keywords": "otonom sürüş, autonomous driving, adas, şerit takip, lane keeping, otonom araç, otomatik park, çarpışma önleme, kör nokta uyarısı"},
            {"isim": "Araç içi güvenlik, konfor ve kullanıcı deneyimi", "keywords": "hava yastığı, airbag, emniyet kemeri, bilgi eğlence sistemi, infotainment, klima, araç koltuğu, sürücü konforu"},
            {"isim": "Raylı ulaşım sistemleri", "keywords": "tren, train, metro, tramvay, demiryolu, railway, vagon, sinyalizasyon, ray hattı, makas"},
            {"isim": "Denizcilik ve su üstü/su altı taşıtları", "keywords": "gemi, ship, tekne, boat, denizaltı, submarine, pervane, propeller, dümen, sonar, tersane, deniz taşımacılığı"},
            {"isim": "Lojistik, filo ve taşımacılık operasyonları", "keywords": "lojistik, logistics, kargo, cargo, konteyner, container, filo yönetimi, rota optimizasyonu, palet, taşıma aracı"}
        ]
    },
    {
        "ana_kategori": "Endüstriyel Üretim ve Sanayi",
        "alt_kategoriler": [
            {"isim": "Üretim makineleri, fabrika otomasyonu ve üretim hatları", "keywords": "montaj hattı, assembly line, konveyör, conveyor, cnc tezgahı, pres makinesi, scada, endüstri 4.0, fabrika otomasyonu"},
            {"isim": "Kalite kontrol, test ve metroloji", "keywords": "kalite kontrol, quality control, hata tespiti, spektrometre, tahribatsız muayene, ndt, ölçüm cihazı, kamera kontrolü, metroloji"},
            {"isim": "Endüstriyel bakım ve operasyon sürekliliği", "keywords": "kestirimci bakım, predictive maintenance, titreşim analizi, yağlama sistemi, yedek parça, arıza teşhisi, bakım yönetimi"},
            {"isim": "Depo, intralojistik ve malzeme taşıma", "keywords": "forklift, vinç, crane, depo otomasyonu, agv, raf sistemi, istifleme, barkod okuyucu, malzeme taşıma"},
            {"isim": "Madencilik, metalurji ve ağır sanayi uygulamaları", "keywords": "madencilik, mining, sondaj, drilling, kazı makinesi, kırıcı, cevher hazırlama, yüksek fırın, çelikhane, dökümhane"},
            {"isim": "Ambalaj, baskı ve dönüştürme sanayii", "keywords": "ambalaj, packaging, paketleme makinesi, etiketleme, dolum tesisi, ofset baskı, matbaa, karton kutu, folyolama"}
        ]
    },
    {
        "ana_kategori": "Enerji, Altyapı ve Kamu Hizmetleri",
        "alt_kategoriler": [
            {"isim": "Yenilenebilir enerji uygulamaları", "keywords": "yenilenebilir enerji, renewable energy, güneş santrali, solar farm, rüzgar parkı, wind farm, jeotermal, biyokütle tesisi, temiz enerji üretimi"},
            {"isim": "Enerji depolama ve şebeke uygulamaları", "keywords": "enerji depolama sistemi, ess, batarya parkı, şebeke dengeleme, frekans regülasyonu, şebeke ölçekli depolama"},
            {"isim": "Elektrik iletim, dağıtım ve güç altyapısı", "keywords": "yüksek gerilim hattı, trafo, şalt sahası, switchyard, izolatör, elektrik sayacı, dağıtım panosu, güç altyapısı"},
            {"isim": "Su, atıksu ve çevresel altyapı", "keywords": "su şebekesi, kanalizasyon, sewer, arıtma tesisi, treatment plant, baraj, pompa istasyonu, çevresel altyapı"},
            {"isim": "Atık yönetimi, geri dönüşüm ve kaynak geri kazanımı", "keywords": "atık yönetimi, waste management, atık ayırma, insineratör, incinerator, geri dönüşüm kutusu, plastik kırma makinesi, kaynak geri kazanımı"},
            {"isim": "Hidrojen ve alternatif enerji altyapısı", "keywords": "hidrojen üretimi, hydrogen production, elektrolizör, electrolyzer, yakıt hücresi istasyonu, hidrojen depolama, sentetik yakıt"},
            {"isim": "Nükleer ve radyasyon temelli altyapılar", "keywords": "nükleer santral, nuclear plant, soğutma kulesi, reaktör kalbi, radiation shielding, radyasyon zırhlama, izotop üretimi"},
            {"isim": "Petrol, doğalgaz, sondaj ve rafineri uygulamaları", "keywords": "petrol, oil, doğalgaz, natural gas, boru hattı, pipeline, rafineri, refinery, matkap ucu, petrokimya tesisi, kompresör istasyonu, lng"}
        ]
    },
    {
        "ana_kategori": "Bilgi Teknolojileri ve Dijital Hizmetler",
        "alt_kategoriler": [
            {"isim": "Kurumsal yazılım ve iş uygulamaları", "keywords": "kurumsal yazılım, enterprise software, erp, crm, insan kaynakları yazılımı, muhasebe sistemi, ofis programı, belge yönetimi"},
            {"isim": "Finansal teknolojiler ve ödeme sistemleri", "keywords": "fintech, pos cihazı, kredi kartı, mobile banking, mobil bankacılık, dijital cüzdan, digital wallet, ödeme sistemi, kripto para borsası"},
            {"isim": "E-ticaret, dijital pazarlama ve müşteri deneyimi", "keywords": "e ticaret, e-commerce, alışveriş sepeti, reklam hedefleme, seo, tavsiye motoru, recommendation engine, online mağaza, chatbot"},
            {"isim": "Siber güvenlik uygulamaları", "keywords": "antivirüs, antivirus, vpn, kimlik doğrulama, ddos koruması, şifre yöneticisi, password manager, güvenlik duvarı, firewall"},
            {"isim": "Veri, analitik ve karar destek uygulamaları", "keywords": "iş zekası, business intelligence, bi, veri görselleştirme, reporting, raporlama, prediktif analiz, dashboard, karar destek"},
            {"isim": "Eğitim teknolojileri ve dijital öğrenme", "keywords": "eğitim teknolojisi, edtech, lms, uzaktan eğitim, e öğrenme, e-learning, akıllı tahta, çevrimiçi sınav, eğitim simülasyonu"},
            {"isim": "Medya, içerik ve dijital platform uygulamaları", "keywords": "video streaming, video akış, oyun motoru, game engine, sosyal medya, content management system, cms, içerik yönetim sistemi, müzik platformu"}
        ]
    },
    {
        "ana_kategori": "Telekomünikasyon Hizmetleri ve Ağ İşletmeciliği",
        "alt_kategoriler": [
            {"isim": "Mobil haberleşme hizmetleri ve telekom operasyonları", "keywords": "gsm operatörü, sim kart, dolaşım, roaming, baz istasyonu yönetimi, mobil ağ operatörü, telekom operasyonu"},
            {"isim": "Sabit haberleşme ve genişbant hizmetleri", "keywords": "adsl, vdsl, fiber internet, ev telefonu, modem, internet servis sağlayıcı, isp, genişbant hizmeti"},
            {"isim": "Uydu haberleşmesi ve bağlantı hizmetleri", "keywords": "vsat, çanak anten, uydu alıcısı, uydu interneti, satellite internet, yörünge operatörü"},
            {"isim": "IoT ve bağlantılı cihaz hizmetleri", "keywords": "m2m sim, akıllı şehir altyapısı, telemetri hizmeti, bağlantılı araç servisi, iot hizmeti"},
            {"isim": "Telekom servis yönetimi ve operasyonel optimizasyon", "keywords": "oss, bss, faturalandırma sistemi, ağ izleme, network monitoring, çağrı merkezi, müşteri destek hattı"}
        ]
    },
    {
        "ana_kategori": "Elektronik ve Akıllı Tüketici Cihazları",
        "alt_kategoriler": [
            {"isim": "Tüketici elektroniği ve akıllı cihazlar", "keywords": "akıllı telefon, smartphone, tablet, dizüstü bilgisayar, laptop, akıllı saat, smart watch, oyun konsolu, fotoğraf makinesi"},
            {"isim": "Görüntü, ekran ve ses sistemleri", "keywords": "televizyon, tv, monitör, projektör, hoparlör, speaker, kulaklık, mikrofon, soundbar"},
            {"isim": "Beyaz eşya ve ev aletleri", "keywords": "buzdolabı, refrigerator, çamaşır makinesi, bulaşık makinesi, fırın, oven, elektrikli süpürge, mikrodalga, ütü"},
            {"isim": "Kişisel bakım, hijyen ve ev içi yardımcı cihazlar", "keywords": "saç kurutma makinesi, tıraş makinesi, akıllı tartı, elektrikli diş fırçası, robot süpürge, kişisel bakım cihazı"},
            {"isim": "Giyilebilir ve taşınabilir cihazlar", "keywords": "akıllı bileklik, fitness takipçisi, akıllı gözlük, taşınabilir şarj cihazı, powerbank, giyilebilir cihaz"}
        ]
    },
    {
        "ana_kategori": "İnşaat, Yapı ve Bina Teknolojileri",
        "alt_kategoriler": [
            {"isim": "Yapı malzemeleri ve yapı kimyasalları", "keywords": "çimento, cement, beton, concrete, tuğla, brick, yalıtım malzemesi, alçı, boya, su yalıtımı, yapıştırıcı"},
            {"isim": "Taşıyıcı sistemler ve yapısal güvenlik", "keywords": "kolon, kiriş, beam, çelik konstrüksiyon, temel, foundation, sismik izolatör, deprem güçlendirme, iskele"},
            {"isim": "Prefabrik, modüler ve cephe sistemleri", "keywords": "prefabrik, prefab, modüler yapı, giydirme cephe, curtain wall, pencere profili, cam balkon, konteyner yapı"},
            {"isim": "İnşaat ekipmanları ve saha makineleri", "keywords": "vinç, crane, ekskavatör, excavator, beton mikseri, asfalt makinesi, saha ekipmanı, kalıp"},
            {"isim": "Tesisat, armatür ve bina içi teknik sistemler", "keywords": "boru tesisatı, plumbing, musluk, faucet, vana, kalorifer peteği, asansör, elevator, yürüyen merdiven, havalandırma kanalı"},
            {"isim": "Akıllı bina ve bina işletim uygulamaları", "keywords": "akıllı bina, smart building, akıllı ev, smart home, termostat, güvenlik kamerası, yangın alarmı, geçiş kontrol sistemi, aydınlatma otomasyonu"}
        ]
    },
    {
        "ana_kategori": "Tarım, Gıda ve Biyokaynak Sistemleri",
        "alt_kategoriler": [
            {"isim": "Tarımsal üretim ve hassas tarım", "keywords": "sera, greenhouse, sulama sistemi, irrigation, tohum, seed, gübre, fertilizer, hidroponik, hassas tarım, tarımsal sensör, rekolte tahmini"},
            {"isim": "Tarım makineleri ve saha ekipmanları", "keywords": "traktör, tractor, biçerdöver, combine harvester, pulluk, mibzer, ilaçlama makinesi, çapa makinesi, tarım dronu"},
            {"isim": "Hayvancılık ve veteriner uygulamaları", "keywords": "sağım makinesi, yemleme sistemi, kuluçka makinesi, veteriner aleti, hayvan takip küpesi, hayvancılık ekipmanı"},
            {"isim": "Gıda işleme ve üretim sistemleri", "keywords": "gıda işleme, food processing, fırınlama makinesi, pastörizasyon, pasteurization, öğütücü, mikser, gıda paketleme, dondurucu, kurutma fırını"},
            {"isim": "Gıda kalite, güvenlik ve izlenebilirlik", "keywords": "gıda analiz cihazı, soğuk zincir, cold chain, barkod takibi, kalite kontrol, food safety, gıda güvenliği, raf ömrü, izlenebilirlik"},
            {"isim": "Fonksiyonel gıda, beslenme ve takviye uygulamaları", "keywords": "probiyotik, probiotic, vitamin, diyet takviyesi, dietary supplement, protein tozu, fonksiyonel gıda, gıda katkı maddesi"}
        ]
    },
    {
        "ana_kategori": "Havacılık, Uzay ve Savunma",
        "alt_kategoriler": [
            {"isim": "Hava araçları ve uçuş sistemleri", "keywords": "uçak, aircraft, helikopter, helicopter, jet motoru, kanat, wing, iniş takımı, aviyonik, avionics, kokpit, uçuş kontrol"},
            {"isim": "İHA, drone ve otonom hava platformları", "keywords": "insansız hava aracı, iha, uav, siha, drone, pervane, uçuş kontrolcüsü, iha kamerası, otonom uçuş"},
            {"isim": "Uzay sistemleri ve uydu platformları", "keywords": "roket, rocket, uzay aracı, spacecraft, yörünge, orbit, itki sistemi, propulsion, uydu, satellite, haberleşme faydalı yükü"},
            {"isim": "Savunma platformları ve görev ekipmanları", "keywords": "zırhlı araç, armored vehicle, tank, denizaltı, silah sistemi, weapon system, füze, missile, mühimmat, ammunition, nişangah, radar"},
            {"isim": "Balistik, koruma ve kritik güvenlik uygulamaları", "keywords": "balistik, ballistic, çelik yelek, balistik kalkan, zırh, armor, kamuflaj, mayın tarama, patlayıcı imha"}
        ]
    },
    {
        "ana_kategori": "Kimya, Malzeme ve Proses Sanayii",
        "alt_kategoriler": [
            {"isim": "Temel kimya ve ara ürün üretimi", "keywords": "petrokimya, petrochemical, asit üretimi, gübre hammaddesi, endüstriyel gaz, industrial gas, amonyak, ammonia, klor, chlorine"},
            {"isim": "Polimer, plastik ve kauçuk sanayii", "keywords": "pvc, polietilen, polyethylene, lastik, tire, enjeksiyon kalıplama, injection molding, granül, ekstrüder, extruder, kauçuk vulkanizasyon"},
            {"isim": "Kaplama, yüzey işlem ve özel malzeme uygulamaları", "keywords": "endüstriyel boya, toz boya, powder coating, teflon kaplama, paslanmazlık kaplaması, yapıştırıcı, adhesive, mastik"},
            {"isim": "Teknik tekstil ve malzeme bazlı endüstriyel uygulamalar", "keywords": "geotekstil, geotextile, yanmaz kumaş, filtre bezi, paraşüt kumaşı, endüstriyel bant, izolasyon örtüsü"}
        ]
    },
    {
        "ana_kategori": "Tekstil, Giyim ve Ayakkabı",
        "alt_kategoriler": [
            {"isim": "Tekstil üretimi ve kumaş teknolojileri", "keywords": "iplik, yarn, dokuma tezgahı, weaving loom, örme, knitting, boyahane, kumaş, fabric, sentetik elyaf, pamuk, terbiye"},
            {"isim": "Hazır giyim ve konfeksiyon uygulamaları", "keywords": "dikiş makinesi, sewing machine, kalıp, pattern, kesim makinesi, kıyafet, garment, fermuar, düğme, ütü presi"},
            {"isim": "Ayakkabı, taban ve saya teknolojileri", "keywords": "ayakkabı tabanı, shoe sole, saya, upper, bağcık, iç taban, insole, poliüretan taban döküm, ayakkabı kalıbı"},
            {"isim": "Moda aksesuarları ve giyim tamamlayıcı bileşenleri", "keywords": "çanta, bag, cüzdan, wallet, kemer, belt, toka, buckle, takı, jewelry, şemsiye, saat kordonu"}
        ]
    },
    {
        "ana_kategori": "Yaşam Alanı ve Tüketici Ürünleri",
        "alt_kategoriler": [
            {"isim": "Mobilya, iç mekân ve dekorasyon uygulamaları", "keywords": "koltuk, sofa, yatak, bed, masa, table, dolap, cabinet, menteşe, çekmece rayı, aydınlatma armatürü, halı"},
            {"isim": "Spor, oyun, oyuncak ve eğlence ürünleri", "keywords": "koşu bandı, treadmill, bisiklet, bicycle, dambıl, masa oyunu, kutu oyunu, peluş oyuncak, lunapark aleti, olta"},
            {"isim": "Evcil hayvan ürün ve bakım uygulamaları", "keywords": "kedi kumu, köpek tasması, akvaryum, kafes, evcil hayvan maması, pet food, evcil hayvan oyuncağı"},
            {"isim": "Kozmetik, kişisel bakım ürünleri ve ev bakım/temizlik ürünleri", "keywords": "krem, cream, parfüm, perfume, makyaj malzemesi, şampuan, shampoo, deterjan, sabun, paspas, bulaşık süngeri"},
            {"isim": "Genel yaşam tarzı ve tüketici ürünleri", "keywords": "bavul, suitcase, valiz, termos, mutfak gereçleri, mutfak bıçağı, saklama kabı, mangal, kamp çadırı"},
            {"isim": "Eğitim, kültür ve hobi amaçlı fiziksel ürünler", "keywords": "kalem, pen, defter, notebook, müzik aleti, piyano, gitar, resim fırçası, kitap ciltleme"}
        ]
    }
]


def get_main_cat_context(tax_item):
    all_keywords = [sub["keywords"] for sub in tax_item["alt_kategoriler"]]
    return f"{tax_item['ana_kategori']} alanındaki teknolojiler ve terimler: {', '.join(all_keywords)}"


def get_sub_cat_context(main_cat_name, sub_cat_dict):
    return f"{main_cat_name} alanında {sub_cat_dict['isim']}. İlgili terimler: {sub_cat_dict['keywords']}"


def format_for_e5_query(texts):
    instruction = (
        "Instruct: Given a Turkish patent title and abstract, "
        "retrieve the most semantically relevant technology or sector category.\n"
        "Query: "
    )
    return [instruction + t for t in texts]


def format_for_e5_passage(texts):
    return [f"passage: {t}" for t in texts]


device = "cuda" if (USE_GPU and torch.cuda.is_available()) else "cpu"
print(f"Kullanılan cihaz: {device.upper()}")

print(f"Model yükleniyor ({MODEL_NAME})...")
model = SentenceTransformer(MODEL_NAME, device=device)

if hasattr(model, "max_seq_length"):
    model.max_seq_length = MAX_SEQ_LENGTH


df["title"] = df["title"].fillna("").astype(str).str.strip()

if "abstract" in df.columns:
    df["abstract"] = df["abstract"].fillna("").astype(str).str.strip()
else:
    df["abstract"] = ""

df = df[(df["title"] != "") | (df["abstract"] != "")].reset_index(drop=True)

df["metin"] = np.where(
    df["abstract"].str.len() > 0,
    df["title"] + ". " + df["abstract"],
    df["title"]
)

print(f"İşlenecek toplam patent/metin sayısı: {len(df)}")


print("Taksonomi bağlamları oluşturuluyor...")

tech_main_texts = format_for_e5_passage([
    get_main_cat_context(item) for item in tech_taxonomy
])

tech_main_embs = model.encode(
    tech_main_texts,
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=BATCH_SIZE
)

tech_sub_embs_dict = {}

for i, item in enumerate(tech_taxonomy):
    sub_texts = format_for_e5_passage([
        get_sub_cat_context(item["ana_kategori"], sub)
        for sub in item["alt_kategoriler"]
    ])

    tech_sub_embs_dict[i] = model.encode(
        sub_texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        batch_size=BATCH_SIZE
    )


sector_main_texts = format_for_e5_passage([
    get_main_cat_context(item) for item in sector_taxonomy
])

sector_main_embs = model.encode(
    sector_main_texts,
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=BATCH_SIZE
)

sector_sub_embs_dict = {}

for i, item in enumerate(sector_taxonomy):
    sub_texts = format_for_e5_passage([
        get_sub_cat_context(item["ana_kategori"], sub)
        for sub in item["alt_kategoriler"]
    ])

    sector_sub_embs_dict[i] = model.encode(
        sub_texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        batch_size=BATCH_SIZE
    )


print("Patent metinleri vektörleştiriliyor...")

patent_texts = format_for_e5_query(df["metin"].tolist())

patent_embs = model.encode(
    patent_texts,
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=BATCH_SIZE,
    show_progress_bar=True
)

print("Hiyerarşik eşleştirmeler yapılıyor...")

sim_tech_main = util.cos_sim(patent_embs, tech_main_embs)
sim_sector_main = util.cos_sim(patent_embs, sector_main_embs)

best_tech_main_scores, best_tech_main_idx = torch.max(sim_tech_main, dim=1)
best_sector_main_scores, best_sector_main_idx = torch.max(sim_sector_main, dim=1)

best_tech_main_idx = best_tech_main_idx.cpu().numpy()
best_sector_main_idx = best_sector_main_idx.cpu().numpy()
best_tech_main_scores = best_tech_main_scores.cpu().numpy()
best_sector_main_scores = best_sector_main_scores.cpu().numpy()

ana_teknolojiler = []
alt_teknolojiler = []
ana_sektorler = []
alt_sektorler = []

for i in range(len(df)):
    p_emb = patent_embs[i].unsqueeze(0)

    t_m_idx = best_tech_main_idx[i]
    ana_teknolojiler.append(tech_taxonomy[t_m_idx]["ana_kategori"])

    sim_tech_sub = util.cos_sim(p_emb, tech_sub_embs_dict[t_m_idx])
    best_t_s_idx = torch.argmax(sim_tech_sub).item()
    alt_teknolojiler.append(
        tech_taxonomy[t_m_idx]["alt_kategoriler"][best_t_s_idx]["isim"]
    )

    s_m_idx = best_sector_main_idx[i]
    ana_sektorler.append(sector_taxonomy[s_m_idx]["ana_kategori"])

    sim_sec_sub = util.cos_sim(p_emb, sector_sub_embs_dict[s_m_idx])
    best_s_s_idx = torch.argmax(sim_sec_sub).item()
    alt_sektorler.append(
        sector_taxonomy[s_m_idx]["alt_kategoriler"][best_s_s_idx]["isim"]
    )


df["Ana_Teknoloji"] = ana_teknolojiler
df["Alt_Teknoloji"] = alt_teknolojiler
df["Ana_Sektor"] = ana_sektorler
df["Alt_Sektor"] = alt_sektorler

df["Ana_Teknoloji_Skor"] = np.round(best_tech_main_scores, 3)
df["Ana_Sektor_Skor"] = np.round(best_sector_main_scores, 3)

df = df.drop(columns=["metin"])

print(f"\nİşlem tamamlandı. Sonuçlar kaydediliyor: {OUTPUT_FILE}")
df.to_excel(OUTPUT_FILE, index=False)
print("Süreç başarıyla sonlandırıldı.")
display(df)

Kullanılan cihaz: CUDA
Model yükleniyor (intfloat/multilingual-e5-large-instruct)...
İşlenecek toplam patent/metin sayısı: 10
Taksonomi bağlamları oluşturuluyor...
Patent metinleri vektörleştiriliyor...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Hiyerarşik eşleştirmeler yapılıyor...

İşlem tamamlandı. Sonuçlar kaydediliyor: output.xlsx
Süreç başarıyla sonlandırıldı.


,patent_no,title,abstract,Ana_Teknoloji,Alt_Teknoloji,Ana_Sektor,Alt_Sektor,Ana_Teknoloji_Skor,Ana_Sektor_Skor
0,PT2028/000001,Makine Öğrenmesi Tabanlı Kestirimci Bakım Sistemi,"Bu buluş, endüstriyel üretim hatlarında kullan...","Mekanik, Mekatronik ve Robotik Sistemler","İmalat yöntemleri, şekillendirme ve montaj tek...",Endüstriyel Üretim ve Sanayi,Endüstriyel bakım ve operasyon sürekliliği,0.882,0.891
1,PT2028/000002,Lityum İyon Batarya Hücreleri İçin Termal Yöne...,"Buluş, elektrikli araç batarya paketlerinde hü...",Enerji ve Çevre Teknolojileri,Enerji depolama ve batarya teknolojileri,"Otomotiv, Mobilite ve Ulaşım",Elektrikli ve hibrit araç sistemleri,0.870,0.860
2,PT2028/000003,IoT Tabanlı Akıllı Sulama ve Toprak Nem İzleme...,"Bu sistem, tarımsal alanlarda toprak nem sensö...","Ağ, İletim ve Bağlantı Teknolojileri",IoT ve bağlantılı cihaz mimarileri,"Tarım, Gıda ve Biyokaynak Sistemleri",Tarımsal üretim ve hassas tarım,0.868,0.900
3,PT2028/000004,Görüntü İşleme Destekli Kalite Kontrol Cihazı,"Buluş, üretim hattındaki ürünleri kamera sensö...","Optik, Fotonik ve Görüntüleme Teknolojileri",Optik algılama ve görüntüleme sistemleri,Endüstriyel Üretim ve Sanayi,"Kalite kontrol, test ve metroloji",0.860,0.880
4,PT2028/000005,Kriptografik Kimlik Doğrulama İçeren Dijital C...,"Bu buluş, mobil ödeme sistemlerinde kullanıcı ...","Yazılım, Veri ve Yapay Zeka","Siber güvenlik, kriptografi ve güvenli bilişim",Bilgi Teknolojileri ve Dijital Hizmetler,Finansal teknolojiler ve ödeme sistemleri,0.859,0.871
5,PT2028/000006,Robot Kol İçin Kuvvet Geri Beslemeli Manipülat...,"Buluş, endüstriyel robot kollarında hassas mon...","Mekanik, Mekatronik ve Robotik Sistemler",Robotik ve otonom fiziksel sistemler,Endüstriyel Üretim ve Sanayi,"Depo, intralojistik ve malzeme taşıma",0.893,0.869
6,PT2028/000007,Güneş Paneli Verimliliğini Artıran İnce Film K...,"Bu buluş, fotovoltaik güneş panellerinde ışık ...",Enerji ve Çevre Teknolojileri,Enerji üretim teknolojileri,"Enerji, Altyapı ve Kamu Hizmetleri",Yenilenebilir enerji uygulamaları,0.863,0.853
7,PT2028/000008,Uzaktan Hasta İzleme İçin Giyilebilir Biyosens...,"Buluş, hastanın kalp ritmi, vücut sıcaklığı ve...",Biyoteknoloji ve Yaşam Bilimleri Teknolojileri,Biyosensör ve biyolojik analiz teknolojileri,"Sağlık, Tıp ve Yaşam Bilimleri",Dijital sağlık ve uzaktan hasta takibi,0.874,0.901
8,PT2028/000009,5G Ağlarında Dinamik Bant Genişliği Tahsisi Yö...,"Bu yöntem, 5G baz istasyonlarında ağ trafiğini...","Ağ, İletim ve Bağlantı Teknolojileri",Ağ yönetimi ve optimizasyon,Telekomünikasyon Hizmetleri ve Ağ İşletmeciliği,Sabit haberleşme ve genişbant hizmetleri,0.901,0.890
9,PT2028/000010,Geri Dönüştürülmüş Polimer Kompozit Yapı Malze...,"Buluş, geri dönüştürülmüş plastik ve cam elyaf...","Kimya, Malzeme ve Yüzey Teknolojileri","Polimer, plastik, kauçuk ve kompozit teknoloji...","İnşaat, Yapı ve Bina Teknolojileri",Yapı malzemeleri ve yapı kimyasalları,0.858,0.866
